In [1]:
import pynq
import pynq.ps
setattr(pynq, 'ps', pynq.ps)
import pynq.lib.video
from pynq import Overlay, allocate
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import time, gc

ol = Overlay("ns_filter_bd_wrapper.bit")
ns = ol.ns_filter_0; dma = ol.axi_dma_0
H, W = 1080, 1920
PATCH_SIZE, HALF, FILTER_SIZE = 7, 3, 49

def mask_and_basis():
    m = np.zeros((7,7), dtype=bool)
    for r,cs in [(0,[3]),(1,[2,3,4]),(2,[1,2,3,4,5]),(3,[1,2,3,4,5]),(4,[1,2,3,4,5]),(5,[2,3,4]),(6,[3])]:
        m[r,cs] = True
    m = m.ravel()
    mi = np.where(m)[0]
    pos = {int(i):k for k,i in enumerate(mi)}
    seen, cols = set(), []
    for i in mi:
        i=int(i)
        if i in seen: continue
        j = m.size-1-i
        c = np.zeros(mi.size, np.float32); c[pos[i]]=1.0
        if j!=i: c[pos[j]]=1.0
        cols.append(c); seen.update([i,j])
    return m, np.stack(cols, axis=1)
mask, basis = mask_and_basis()
K = basis.shape[1]
mi = np.where(mask)[0]
pairs = [[int(mi[r]) for r in np.where(basis[:,k]==1.0)[0]] for k in range(K)]
w_vec = basis.sum(0).astype(np.float64)

def reset_dma():
    mm = dma.mmio
    mm.write(0x00,0x4); mm.write(0x30,0x4); time.sleep(0.01)
    mm.write(0x00,0x1); mm.write(0x30,0x1)
    dma.sendchannel._first_transfer = True
    dma.recvchannel._first_transfer = True

def load_all_ru_taps(all_taps, all_norms):
    for r in range(5):
        for c in range(8):
            bank = r*8 + c
            ns.write(0x3C, bank << 8)
            for i in range(12):
                ns.write(i*4, int(all_taps[bank][i]) & 0x7F)
            ns.write(0x30, int(all_norms[bank]) & 0xFFFF)

def run_hw_frame(sr_uint8):
    """Modified: polls TLAST directly instead of PYNQ wait() to avoid hang."""
    ns.write(0x34, sr_uint8.shape[1])
    ns.write(0x38, sr_uint8.shape[0])
    in_buf  = allocate(shape=(H*W,),     dtype=np.uint32)
    out_buf = allocate(shape=((H-6)*W,), dtype=np.uint32)
    in_buf[:] = sr_uint8.flatten().astype(np.uint32)
    reset_dma()
    t0 = time.time()
    dma.recvchannel.transfer(out_buf)
    ns.write(0x3C, 0x1)
    dma.sendchannel.transfer(in_buf)
    timeout = 5.0
    while ((ns.read(0x04) >> 3) & 1) == 0:
        if time.time() - t0 > timeout:
            print(f"WARN: TLAST timeout, beats={ns.read(0x00)}")
            break
    time.sleep(0.05)
    dt = (time.time()-t0)*1000
    hw = (out_buf & 0xFF).astype(np.uint8).copy()
    in_buf.freebuffer(); out_buf.freebuffer(); del in_buf, out_buf; gc.collect()
    return hw, dt

prelr = np.fromfile("/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_PRELR/RANGE_1/frame_014_qp160_prelr.yuv", dtype=np.uint8).reshape(H,W)
hr    = np.fromfile("/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_HR/frame_014.y", dtype=np.uint8).reshape(H,W)
print("setup ok")

setup ok


In [2]:
def train_one_ru(sr_ru, hr_ru, sr_padded_slab):
    win = sliding_window_view(sr_padded_slab, (PATCH_SIZE, PATCH_SIZE))
    Xh, Xw = sr_ru.shape
    patches = win[:Xh, :Xw].reshape(-1, FILTER_SIZE)
    Xm = patches[:, mask].astype(np.float64) / 255.0
    y  = hr_ru.astype(np.float64).ravel() / 255.0
    XB = Xm @ basis
    A  = XB.T @ XB + 0.01 * np.eye(K)
    b  = XB.T @ y
    KKT = np.zeros((K+1, K+1)); KKT[:K,:K]=A; KKT[:K,K]=w_vec; KKT[K,:K]=w_vec
    try:
        f_u = np.linalg.solve(KKT, np.concatenate([b, [1.0]]))[:K]
    except np.linalg.LinAlgError:
        return np.array([0]*11 + [32], np.int8), 2048
    f_c = basis @ f_u
    yy = float(y @ y); N = len(y)
    A_K = XB.T @ XB; b_K = XB.T @ y
    rep = np.array([int(np.argmax(basis[:,k])) for k in range(K)])
    f_uc = f_c[rep].astype(np.float64)
    scale = 63/max(abs(f_uc).max(), 1e-12)
    f_int = np.clip(np.round(f_uc*scale), -63, 63).astype(np.int64)
    def mse(fi):
        fr = fi/scale; s = float(w_vec@fr)
        if abs(s)<1e-6: return float('inf')
        fs = fr/s; return float((yy - 2*fs@b_K + fs@A_K@fs)/N)
    best = mse(f_int)
    for _ in range(20):
        imp=False
        for k in range(K):
            bd=0
            for d in (-1,1):
                t = f_int.copy(); t[k]=np.clip(t[k]+d,-63,63)
                if t[k]==f_int[k]: continue
                m=mse(t)
                if m<best-1e-12: best,bd=m,d
            if bd: f_int[k]=np.clip(f_int[k]+bd,-63,63); imp=True
        if not imp: break
    ws = int(2*f_int[:11].sum() + f_int[11])
    norm = int(round(65536/ws)) & 0xFFFF if ws!=0 else 0
    return f_int.astype(np.int8), norm

print("Training 40 per-RU tap sets...")
t0 = time.time()
sr_pad_uint = np.pad(prelr, HALF, mode="symmetric")
all_taps = []; all_norms = []
for r in range(5):
    r0 = r*256; r1 = min(r0+256, H)
    for c in range(8):
        c0 = c*256; c1 = min(c0+256, W)
        slab = sr_pad_uint[r0:r1+2*HALF, c0:c1+2*HALF]
        f_int, norm = train_one_ru(prelr[r0:r1, c0:c1], hr[r0:r1, c0:c1], slab)
        all_taps.append(f_int)
        all_norms.append(norm)
print(f"train time: {time.time()-t0:.1f}s   (40 tap sets ready)")

Training 40 per-RU tap sets...
train time: 32.0s   (40 tap sets ready)


In [3]:
load_all_ru_taps(all_taps, all_norms)
ns.write(0x3C, 0x2)                         # flip
hw_flat, dt = run_hw_frame(prelr)
print(f"HW: {dt:.1f} ms, beats={ns.read(0x00)}, tlast={(ns.read(0x04)>>3)&1}")

# Python golden: per-RU apply
print("Applying Python golden per-RU...")
t0 = time.time()
src = np.pad(prelr, HALF, mode="symmetric").astype(np.int32)
py_full = np.zeros((H, W), dtype=np.uint8)
for r in range(5):
    r0 = r*256; r1 = min(r0+256, H)
    for c in range(8):
        c0 = c*256; c1 = min(c0+256, W)
        bank = r*8 + c
        taps, norm = all_taps[bank], all_norms[bank]
        acc = np.zeros((r1-r0, c1-c0), dtype=np.int64)
        for k, positions in enumerate(pairs):
            ps = np.zeros((r1-r0, c1-c0), dtype=np.int32)
            for p in positions:
                dr, dc = p//7-HALF, p%7-HALF
                ps += src[HALF+r0+dr:HALF+r1+dr, HALF+c0+dc:HALF+c1+dc]
            acc += int(taps[k]) * ps
        py_full[r0:r1, c0:c1] = np.clip((acc*int(norm) + (1<<15))>>16, 0, 255).astype(np.uint8)
        del acc
print(f"apply time: {time.time()-t0:.1f}s")

# Aligned compare
start = 3*W + 3
exp = py_full.flatten()[start : start + hw_flat.size]
diff = hw_flat.astype(np.int16) - exp.astype(np.int16)
core = diff[:-20]
print(f"\nHW vs Python golden (multi-RU, aligned):")
print(f"  max |diff|:  {int(np.abs(core).max())}")
print(f"  mean |diff|: {float(np.abs(core).mean()):.4f}")
print(f"  exact match: {float((core==0).mean())*100:.2f}%")

def psnr(a,b):
    mse=float(np.mean((a.astype(np.int32)-b.astype(np.int32))**2))
    return float('inf') if mse==0 else 10*np.log10(255**2/mse)
hr_exp = hr.flatten()[start : start + hw_flat.size]
prelr_exp = prelr.flatten()[start : start + hw_flat.size]
print(f"\nPSNR vs HR:")
print(f"  pre-LR:          {psnr(prelr_exp, hr_exp):.3f} dB")
print(f"  HW multi-RU:     {psnr(hw_flat, hr_exp):.3f} dB")
print(f"  Python multi-RU: {psnr(exp, hr_exp):.3f} dB")

HW: 84.0 ms, beats=2062084, tlast=1
Applying Python golden per-RU...
apply time: 4.2s

HW vs Python golden (multi-RU, aligned):
  max |diff|:  22
  mean |diff|: 0.0141
  exact match: 99.36%

PSNR vs HR:
  pre-LR:          26.850 dB
  HW multi-RU:     26.970 dB
  Python multi-RU: 26.971 dB
